<a href="https://colab.research.google.com/github/adishup/gen-ai-lab/blob/main/experiment%206.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers sentencepiece accelerate

In [ ]:
# ============================================================
# EXPERIMENT 6
# RETRIEVAL-AUGMENTED GENERATION (RAG)
# USING VECTOR DATABASE
# ============================================================

import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


print("=" * 60)
print("EXPERIMENT 6 - RAG USING VECTOR DATABASE")
print("=" * 60)


# ============================================================
# 1. DEVICE
# ============================================================

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("\nGPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("\nUsing CPU")


# ============================================================
# 2. KNOWLEDGE BASE
# ============================================================

documents = [

    """
    Retrieval-Augmented Generation, commonly called RAG,
    combines information retrieval with text generation.
    A RAG system first retrieves relevant information from
    an external knowledge base and then provides that
    information as context to a language model to generate
    an answer.
    """,

    """
    Vector databases store numerical vector representations
    called embeddings. They allow systems to search for
    documents that are semantically similar to a query.
    FAISS is a library used for efficient similarity search
    over dense vectors.
    """,

    """
    Generative Artificial Intelligence is a branch of
    Artificial Intelligence that can create new content
    such as text, images, audio, video and computer programs.
    """,

    """
    Large Language Models are transformer-based models
    trained on large collections of text. They can perform
    tasks such as text generation, summarization, translation,
    question answering and conversation.
    """,

    """
    Fine-tuning adapts a pretrained machine learning model
    to a specific task or domain by training it on a smaller
    domain-specific dataset.
    """
]


print("\nKnowledge base created.")
print("Number of documents:", len(documents))


# ============================================================
# 3. LOAD EMBEDDING MODEL
# ============================================================

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")


# ============================================================
# 4. CREATE EMBEDDINGS
# ============================================================

print("\nCreating document embeddings...")

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

document_embeddings = document_embeddings.astype(
    "float32"
)

# Normalize embeddings
faiss.normalize_L2(
    document_embeddings
)


# ============================================================
# 5. CREATE FAISS DATABASE
# ============================================================

dimension = document_embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    document_embeddings
)

print("Vector database created successfully.")


# ============================================================
# 6. LOAD FLAN-T5
# ============================================================

print("\nLoading FLAN-T5 model...")

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

model = model.to(device)
model.eval()

print("FLAN-T5 model loaded successfully.")


# ============================================================
# 7. RETRIEVE RELEVANT DOCUMENTS
# ============================================================

def retrieve_documents(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = query_embedding.astype(
        "float32"
    )

    faiss.normalize_L2(
        query_embedding
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for i in range(top_k):

        document_index = indices[0][i]

        score = scores[0][i]

        results.append({
            "document": documents[document_index].strip(),
            "score": float(score)
        })

    return results


# ============================================================
# 8. GENERATE ANSWER
# ============================================================

def generate_answer(query, retrieved_documents):

    context = "\n\n".join(
        item["document"]
        for item in retrieved_documents
    )

    prompt = f"""
You are a question answering system.

Use the CONTEXT to answer the QUESTION.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    input_ids = inputs["input_ids"].to(device)

    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():

        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=80,
            num_beams=5,
            do_sample=False,
            repetition_penalty=1.1
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return answer.strip()


# ============================================================
# 9. ASK QUESTION
# ============================================================

print("\n" + "=" * 60)
print("RAG QUESTION ANSWERING")
print("=" * 60)

query = input(
    "\nEnter your question: "
)


# ============================================================
# 10. RETRIEVE DOCUMENTS
# ============================================================

results = retrieve_documents(
    query,
    top_k=3
)


# ============================================================
# 11. DISPLAY RETRIEVED DOCUMENTS
# ============================================================

print("\n" + "=" * 60)
print("RETRIEVED DOCUMENTS")
print("=" * 60)

for number, result in enumerate(
    results,
    start=1
):

    print(
        f"\nDocument {number}"
    )

    print(
        result["document"]
    )

    print(
        "Similarity Score:",
        round(result["score"], 4)
    )


# ============================================================
# 12. GENERATE ANSWER
# ============================================================

answer = generate_answer(
    query,
    results
)


# ============================================================
# 13. DISPLAY ANSWER
# ============================================================

print("\n" + "=" * 60)
print("GENERATED ANSWER")
print("=" * 60)

print(answer)


# ============================================================
# 14. COMPLETION
# ============================================================

print("\n" + "=" * 60)
print("EXPERIMENT 6 COMPLETED SUCCESSFULLY")
print("=" * 60)

EXPERIMENT 6 - RAG USING VECTOR DATABASE

Using CPU

Knowledge base created.
Number of documents: 5

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.

Creating document embeddings...
Vector database created successfully.

Loading FLAN-T5 model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 model loaded successfully.

RAG QUESTION ANSWERING

Enter your question: What is Retrieval-Augmented Generation?

RETRIEVED DOCUMENTS

Document 1
Retrieval-Augmented Generation, commonly called RAG,
    combines information retrieval with text generation.
    A RAG system first retrieves relevant information from
    an external knowledge base and then provides that
    information as context to a language model to generate
    an answer.
Similarity Score: 0.6568

Document 2
Generative Artificial Intelligence is a branch of
    Artificial Intelligence that can create new content
    such as text, images, audio, video and computer programs.
Similarity Score: 0.3321

Document 3
Fine-tuning adapts a pretrained machine learning model
    to a specific task or domain by training it on a smaller
    domain-specific dataset.
Similarity Score: 0.3207

GENERATED ANSWER
combines information retrieval with text generation

EXPERIMENT 6 COMPLETED SUCCESSFULLY
